# Unit 10 practical: serve a real model and compare with the roofline
> **Status: written but NOT executed** while the course was authored (no GPU or model there). The model exists on the Hugging Face hub (checked).
> Use a Colab GPU (T4) for the speed measurements; the arithmetic cells run anywhere.

You will: (1) recompute the model's **parameter count and KV cache** from the loaded configuration and check them against PyTorch's own count, (2) measure **tokens per second** at several batch sizes, (3) compare with the **roofline prediction** using your GPU's bandwidth, and (4) wrap generation in the course's **web service** (FastAPI).

In [ ]:
!pip -q install transformers accelerate fastapi uvicorn requests
!wget -q https://krunalpanchal1995.github.io/banu.github.io/downloads/u10/u10-code.zip -O u10.zip && unzip -q -o u10.zip
import sys; sys.path.insert(0, "u10-code/codes/serving/serving_np"); import serving_np as S
import torch, time, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
name = "Qwen/Qwen2.5-0.5B-Instruct"; tok = AutoTokenizer.from_pretrained(name); model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16, device_map="auto")

## 1. Recompute the counts (Section 3.1) and compare with PyTorch

In [ ]:
cfg = model.config.to_dict(); cfg.setdefault("attention_bias", True)
b = S.param_breakdown(cfg); torch_count = sum(p.numel() for p in model.parameters())      # tied embeddings are counted once by parameters()
print("computed", b["total"], " torch", torch_count, " hub", S.hub_total(), " equal:", b["total"] == torch_count)
print("KV bytes per token:", S.kv_bytes_per_token(cfg), " weights MB:", S.weight_bytes(b["total"], "bf16") / 1e6)

## 2. Measure decode speed against batch size

In [ ]:
def tokens_per_second(batch, new_tokens=64, prompt="Explain what a KV cache is."):
    ids = tok([prompt] * batch, return_tensors="pt", padding=True).to(model.device)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = model.generate(**ids, max_new_tokens=new_tokens, do_sample=False, min_new_tokens=new_tokens)
    torch.cuda.synchronize(); dt = time.perf_counter() - t0
    return batch * new_tokens / dt
res = {b: tokens_per_second(b) for b in (1, 2, 4, 8, 16, 32)}
print(res)

## 3. Compare with the roofline (Section 3.3.1)
Look up your GPU's memory bandwidth (for a T4 the specification is about 320 GB/s: **check the vendor sheet**) and compute the predicted memory-bound speed `S.decode_throughput(...)`. Real systems reach only a fraction of the roofline: the Python overhead of `generate` matters for a model this small.

In [ ]:
BW = 320e9; FL = 65e12        # replace with your GPU's numbers
n = S.hub_total(); w = S.weight_bytes(n, "bf16"); kv = S.kv_bytes_per_token(cfg) * 100
for b_, measured in res.items(): print(b_, "measured", round(measured), "tokens/s   roofline", round(S.decode_throughput(n, w, kv, b_, BW, FL)))

## 4. Put it behind a web service (Section 4.2)
Copy `app_fastapi.py` from the zip, replace the extractive answerer with a call to `model.generate`, keep the validation and the abstention rule, run `uvicorn`, and measure p50 and p95 with `bench_service.py`. Compare with the extractive service's 1.7 ms: how much of the latency is now the model?

## Your turn
1. Load the model in 8-bit or 4-bit (`bitsandbytes`) and repeat step 2. Does the speed rise as the roofline predicts (3.75× at 4 bits)?
2. Measure time to first token for prompts of 100, 1000 and 4000 tokens (prefill is compute-bound).
3. Fill the GPU with batch size until you run out of memory and compare with `S.max_concurrent_sequences`.